# A2.5 · Delegation that survives audit

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

Builds on **[A2.4 · The NHI governance gap](https://spbreed.github.io/cyber-commons/lessons/A2.4.html)**.

| | |
|---|---|
| Open-source tooling | Keycloak, RFC 8693 |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Now we build the thing A2.1 pointed at and A2.3 proved we need:
**on-behalf-of delegation**, standardised as [RFC 8693 OAuth 2.0 Token
Exchange](https://datatracker.ietf.org/doc/html/rfc8693).

The idea is small. An actor presents a token it holds and asks for a new one for
a different actor. The issuer returns a token with:

- **`sub`** — unchanged. The action is still *for* the original principal.
- **`actor`** — the new holder.
- **`act`** — a nested claim recording who presented the token, and who
  presented it to *them*, all the way back.

Two rules make the result auditable, and both must hold:

1. **Subset of what was presented.** You cannot hand on authority you were not
   given.
2. **Within the new actor's own ceiling.** You cannot hand on authority the
   recipient may never hold, even if the caller offered it. (This is A3.1's
   ceiling, doing its second job.)

Drop either rule and the chain still *looks* correct — every token parses, every
call succeeds — which is precisely why this needs a test rather than a review.

## 2 · Demo — a three-hop chain that narrows at every step

Dana asks for a fix. The orchestrator delegates to a patch agent, which delegates to a deploy agent. Watch the scopes shrink.

In [ ]:
import hashlib, json, time
from dataclasses import dataclass, field

# A3.1's ceilings: what each actor may EVER hold.
CEILINGS = {
    "dana@corp":    {"repo:read", "repo:write", "deploy:prod", "secrets:read"},
    "orchestrator": {"repo:read", "repo:write", "deploy:prod"},
    "patch-agent":  {"repo:read", "repo:write"},
    "deploy-agent": {"repo:read", "deploy:prod"},
    "triage-agent": {"repo:read"},
}

class DelegationError(Exception):
    """Refusing to widen is the feature."""

@dataclass
class Token:
    sub: str
    actor: str
    scopes: set
    act: dict = None
    issued: float = field(default_factory=time.time)
    ttl: float = 300

    @property
    def expired(self): return time.time() - self.issued > self.ttl

    def chain(self):
        out, node = [], self.act
        while node:
            out.append(node["actor"]); node = node.get("act")
        c = list(reversed(out)) + [self.actor]
        if c[0] != self.sub:
            c.insert(0, self.sub)
        return c

    def fingerprint(self):
        blob = json.dumps({"sub": self.sub, "actor": self.actor,
                           "scopes": sorted(self.scopes), "act": self.act},
                          sort_keys=True)
        return hashlib.sha256(blob.encode()).hexdigest()[:12]

    def describe(self):
        return (f"{' → '.join(self.chain()):48s}\n"
                f"      scopes {sorted(self.scopes)}   fp={self.fingerprint()}")

def mint(principal, scopes=None):
    ceiling = CEILINGS[principal]
    want = set(scopes) if scopes else set(ceiling)
    if not want <= ceiling:
        raise DelegationError(f"{principal} cannot hold {sorted(want - ceiling)}")
    return Token(sub=principal, actor=principal, scopes=want)

def exchange(presented, new_actor, scopes):
    """One RFC 8693 hop. Both narrowing rules live here and nowhere else."""
    if presented.expired:
        raise DelegationError("presented token has expired")
    scopes = set(scopes)
    if not scopes <= presented.scopes:                       # rule 1
        raise DelegationError(
            f"widening refused: {sorted(scopes - presented.scopes)} is not in the "
            f"presented token {sorted(presented.scopes)}")
    ceiling = CEILINGS.get(new_actor, set())
    if not scopes <= ceiling:                                # rule 2
        raise DelegationError(
            f"widening refused: {new_actor} may never hold "
            f"{sorted(scopes - ceiling)} (ceiling {sorted(ceiling)})")
    return Token(sub=presented.sub, actor=new_actor, scopes=scopes,
                 act={"actor": presented.actor, "act": presented.act},
                 ttl=min(presented.ttl, 300))

dana  = mint("dana@corp", {"repo:read", "repo:write", "deploy:prod"})
orch  = exchange(dana, "orchestrator", {"repo:read", "repo:write", "deploy:prod"})
patch = exchange(orch, "patch-agent",  {"repo:read", "repo:write"})
ship  = exchange(patch, "deploy-agent", {"repo:read"})

for t in (dana, orch, patch, ship):
    print(t.describe())

## 3 · Demo — the resource server can finally answer the question

This is the payoff. GitHub (or any downstream) receives the last token and can record a truthful, complete line.

In [ ]:
def resource_server(token, required_scope):
    if token.expired:
        return {"allowed": False, "why": "token expired"}
    if required_scope not in token.scopes:
        return {"allowed": False,
                "why": f"needs {required_scope}, holds {sorted(token.scopes)}"}
    return {"allowed": True,
            "audit": f"{token.actor} performed {required_scope} on behalf of "
                     f"{token.sub} via {' → '.join(token.chain()[1:-1]) or 'direct'}",
            "chain": token.chain()}

for tok, scope in ((patch, "repo:write"), (ship, "repo:write"), (ship, "repo:read")):
    r = resource_server(tok, scope)
    print(f"{tok.actor:14s} wants {scope:12s} → {'ALLOW' if r['allowed'] else 'DENY '}")
    print(f"   {r.get('audit') or r['why']}")

## 4 · Where it breaks — three ways, all refused

Now the attacks. Each of these is something a real integration will attempt, usually by accident.

In [ ]:
attacks = [
 ("widen beyond the presented token",
  lambda: exchange(ship, "deploy-agent", {"deploy:prod"})),
 ("widen beyond the actor's own ceiling",
  lambda: exchange(dana, "triage-agent", {"repo:write"})),
 ("replay an expired token",
  lambda: exchange(Token("dana@corp", "patch-agent", {"repo:write"}, ttl=-1),
                   "deploy-agent", {"repo:write"})),
]
for name, fn in attacks:
    try:
        fn()
        print(f"GRANTED  {name}   ← this must not happen")
    except DelegationError as e:
        print(f"REFUSED  {name}\n         {e}")

## 5 · The anti-pattern, for contrast

Impersonation produces a token that works perfectly and destroys the audit trail — the A2.3 failure, now visible next to the correct version.

In [ ]:
def impersonate(principal, actor, scopes):
    """No act claim. The agent simply becomes the human."""
    return Token(sub=principal, actor=principal, scopes=set(scopes), act=None)

bad = impersonate("dana@corp", "patch-agent", {"repo:write"})
print("delegated    :", " → ".join(patch.chain()))
print("impersonated :", " → ".join(bad.chain()), "  ← the agent is invisible")
print("\nresource server sees:")
print("   delegated    :", resource_server(patch, "repo:write")["audit"])
print("   impersonated :", resource_server(bad, "repo:write")["audit"])

In [ ]:
# Verify: property-test the invariant over random chains.
import random
random.seed(11)
actors = [a for a in CEILINGS if a != "dana@corp"]
violations, built = 0, 0
for _ in range(1500):
    tok = mint("dana@corp")
    for _ in range(random.randint(1, 4)):
        nxt = random.choice(actors)
        want = set(random.sample(sorted(tok.scopes),
                                 k=random.randint(0, len(tok.scopes))))
        try:
            new = exchange(tok, nxt, want)
        except DelegationError:
            continue
        if not new.scopes <= tok.scopes or not new.scopes <= CEILINGS[nxt]:
            violations += 1
        tok = new; built += 1
print(f"{built} successful hops across 1500 random chains — widening violations: {violations}")
assert violations == 0
print("Invariant holds: authority can only shrink, on every path.")

## 6 · The review, as a skill

Two narrowing rules decide whether a delegation is safe, and the common failure is checking one of them. **Subset-of-presented** alone lets a highly privileged user hand an agent authority the agent should never hold. **Within-actor-ceiling** alone lets an agent exceed the user who asked. Neither is sufficient; the skill requires both, and its contract has a field for each so a review cannot quietly skip one.

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

In [ ]:
# skills/identity/agent-identity-review/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: agent-identity-review
description: >-
  Review how an agent authenticates and how a user's authority is delegated to
  it, including token exchange, on-behalf-of chains, service accounts and
  non-human identity. Use when asked who an agent is calling as, whether a
  delegation is auditable, why a downstream system sees the wrong principal, or
  how to scope an agent's credentials.
allowed-tools: Read, Grep, Glob
---

# Agent identity and delegation review

Three identities are in play whenever an agent acts, and most incidents come
from collapsing them:

- the **user** whose request started the work
- the **agent** doing the work — a workload identity, not a person
- the **actor chain** connecting them, which is what an auditor needs

A service account shared by every agent answers "what ran" and destroys "for
whom". That is the gap non-human identity exists to close.

## When to use this

Reviewing an agent's auth design, an OBO/token-exchange implementation, a
gateway that fronts an agent, or any log where you cannot tell which user
caused an action.

## Procedure

**1 — Name the three identities.** For the flow under review, write down the
user principal, the workload identity, and where each is asserted. If the
workload identity is a long-lived shared secret, that is finding one.

**2 — Check the delegation narrows.** In RFC 8693 token exchange the issued
token must satisfy **both** rules:

- **subset of presented** — never more scope than the incoming token carried
- **within the actor's ceiling** — never more than the agent is itself allowed

Either rule alone is insufficient. Subset-only lets a highly privileged user
hand an agent authority the agent should never hold; ceiling-only lets an agent
exceed the user who asked. Test both directions explicitly.

**3 — Check the chain is preserved and not duplicated.** The `act` chain should
read `user → agent`, once. A chain that repeats the principal
(`alice → alice → agent`) usually means the head was appended twice, and it
breaks any audit query that counts hops.

**4 — Find where OBO stops.** Some downstream systems cannot consume a
delegated token — legacy databases, vendor APIs, anything with a static
credential. Identify each, and require a **choke point**: a gateway that holds
the credential, enforces per-user authorisation *before* the call, and logs the
original principal. The credential must not be reachable by the agent directly.

**5 — Check expiry and revocation.** How long is the delegated token valid, and
what stops it being replayed after the user's session ends? Just-in-time
authority that outlives the task is standing authority with extra steps.

**6 — Check the log answers the audit question.** Pick a real question — "which
user caused this row to be deleted?" — and try to answer it from the logs alone.
If you cannot, the delegation is not auditable regardless of how it is built.

## Output contract

```json
{
  "identities": {"user": "str", "workload": "str", "assertion": "str"},
  "delegation": {"mechanism": "obo|impersonation|shared_service_account|none",
                 "subset_of_presented": true, "within_actor_ceiling": true,
                 "chain": ["principal", "..."], "chain_wellformed": true,
                 "ttl_seconds": 0, "revocable": true},
  "chokepoints": [{"downstream": "str", "reason": "str",
                   "enforced_at": "gateway|service|none",
                   "credential_reachable_by_agent": false}],
  "audit": {"question": "str", "answerable_from_logs": true},
  "findings": [{"issue": "str", "severity": "critical|high|medium|low", "fix": "str"}]
}
```

## Failure modes

- **Checking only one narrowing rule.** Both, every time.
- **Accepting a shared service account because it is "internal".** Internal is
  a network property; it says nothing about attribution.
- **Treating the gateway as optional** where the downstream cannot do OBO. It
  is the only place authorisation can happen.
- **Measuring delegation by whether the call succeeded.** A call that succeeds
  with too much scope is the failure being looked for.
"""

meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
contract = contract_of(body)

user  = mint("dana@corp", {"repo:read", "repo:write", "deploy:prod"})
agent = exchange(user,  "orchestrator", {"repo:read", "repo:write", "deploy:prod"})
leaf  = exchange(agent, "patch-agent",  {"repo:read", "repo:write"})

presented = set(agent.scopes)
review = {
 "identities": {"user": leaf.sub, "workload": leaf.actor,
                "assertion": "RFC 8693 token exchange"},
 "delegation": {
   "mechanism": "obo",
   "subset_of_presented": leaf.scopes <= presented,
   "within_actor_ceiling": leaf.scopes <= CEILINGS[leaf.actor],
   "chain": leaf.chain(),
   # the head must appear once: alice -> alice -> agent breaks every audit
   # query that counts hops
   "chain_wellformed": len(leaf.chain()) == len(set(leaf.chain())),
   "ttl_seconds": leaf.ttl, "revocable": True},
 "chokepoints": [
   {"downstream": "legacy reporting DB", "reason": "cannot consume a delegated token",
    "enforced_at": "gateway", "credential_reachable_by_agent": False}],
 "audit": {"question": "which user caused this row to be deleted?",
           "answerable_from_logs": True},
 "findings": [],
}
if not review["delegation"]["within_actor_ceiling"]:
    review["findings"].append({"issue": "issued scope exceeds the actor ceiling",
                               "severity": "critical", "fix": "intersect with the ceiling"})

problems = check(review, contract)
print(f"conformance: {len(problems)} problem(s)")
for p in problems: print("   ", p)
assert not problems, problems

d = review["delegation"]
print(f"\nchain            : {' -> '.join(d['chain'])}")
print(f"subset of presented : {d['subset_of_presented']}")
print(f"within ceiling      : {d['within_actor_ceiling']}")
print(f"chain well-formed   : {d['chain_wellformed']}")
assert d["subset_of_presented"] and d["within_actor_ceiling"]
assert d["chain_wellformed"]

## 7 · Where it breaks — checking only one rule

A privileged user asks a limited agent to do something.

In [ ]:
privileged = mint("dana@corp", CEILINGS["dana@corp"])   # includes secrets:read
requested  = {"repo:read", "secrets:read"}

# A gateway that only checks "is this a subset of what the user presented?" --
# the check exchange() does first, in isolation from the one it does second.
subset_only = requested <= privileged.scopes
# The rule it skipped.
ceiling_ok  = requested <= CEILINGS["patch-agent"]

print(f"user presents        : {sorted(privileged.scopes)}")
print(f"requested for agent  : {sorted(requested)}")
print(f"patch-agent ceiling  : {sorted(CEILINGS['patch-agent'])}")
print(f"\nsubset-of-presented  : {subset_only}")
print(f"within-actor-ceiling : {ceiling_ok}")
print()
print("Subset-only says yes. The agent would hold secrets:read because the")
print("*user* could have read secrets - authority the agent is never allowed")
print("to hold, delegated legitimately, and it will look correct in the log.")
print()
print("Both rules, every time. The intersection is the only safe issue.")
safe = requested & CEILINGS["patch-agent"]
print(f"correct issued scope : {sorted(safe)}")
assert subset_only and not ceiling_ok, "this is the exact gap the second rule closes"
assert safe < requested

## What you just proved

Four tokens print with strictly narrowing scopes and readable chains ending in `dana@corp → orchestrator → patch-agent → deploy-agent`. The resource server allows `patch-agent` a write, denies `deploy-agent` the same write, and produces a truthful audit line naming both the actor and the principal. All three attacks are refused with the rule that refused them. The impersonated token's chain contains only `dana@corp`. The property test reports zero widening violations.

## Your turn

Run these same four scenarios against real Keycloak with token exchange enabled. The properties should hold identically — and if your realm allows the second one (widening past the actor's ceiling), that is a live finding, because Keycloak will happily issue it if the client is configured permissively.

---

**Next → [A2.6 · The agentic gateway](https://spbreed.github.io/cyber-commons/lessons/A2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*